# MiniEngine on Google Colab

End-to-end runner for the **CS349D MiniEngine** repo. This notebook:

1. Verifies a GPU is attached
2. Clones the (private) GitHub repo using a Personal Access Token
3. Installs dependencies
4. Launches the OpenAI-compatible server in the background
5. Sends a sample chat request to verify it works
6. (Optional) Runs a quick serving benchmark
7. Cleanly shuts the server down

## Before you start

**Enable GPU**: `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 is free and is enough for `Qwen/Qwen3-4B-Instruct-2507` in bf16).

**Create a GitHub PAT** (only needed if your repo is private):
1. Go to https://github.com/settings/tokens?type=beta
2. Generate new token (fine-grained) → Resource owner: `bwathomas`
3. Repository access: **Only select repositories** → `MAST-miniengine-BWT`
4. Permissions → Repository permissions → **Contents: Read-only**
5. Copy the token (starts with `github_pat_…`). You'll paste it below.

## 1. Verify GPU


In [ ]:
!nvidia-smi

## 2. Clone the repository

Running the next cell will prompt you for your GitHub username and PAT (input is hidden). The token is used only for the clone and is **not** persisted to disk.

If your repo is **public**, you can leave the token blank.

In [ ]:
import os
import shutil
import subprocess
from getpass import getpass
from urllib.parse import quote

GITHUB_USER = "bwathomas"  # @param {type:"string"}
REPO_NAME   = "MAST-miniengine-BWT"  # @param {type:"string"}
BRANCH      = "MAST-miniengine-BWT"  # @param {type:"string"}

WORKDIR = f"/content/{REPO_NAME}"

token = getpass("GitHub Personal Access Token (leave blank if repo is public): ").strip()

if os.path.isdir(WORKDIR):
    print(f"Removing existing checkout at {WORKDIR} \u2026")
    shutil.rmtree(WORKDIR)

if token:
    auth = f"{GITHUB_USER}:{quote(token, safe='')}@"
else:
    auth = ""
clone_url = f"https://{auth}github.com/{GITHUB_USER}/{REPO_NAME}.git"

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, WORKDIR],
    check=True,
)
del token

%cd $WORKDIR
!git log --oneline -5

## 3. Install dependencies

Installs `miniengine` itself plus the benchmark extras. PyTorch is preinstalled on Colab so this is mostly small packages.

In [ ]:
!pip install -q -e .
!pip install -q -e ".[bench]" datasets requests

### (Optional) Persist the HuggingFace model cache to Google Drive

Qwen3-4B weights are ~8 GB. Colab wipes `/root/.cache` between sessions, so subsequent runs would re-download. Mount Drive and point `HF_HOME` there to cache across sessions. **Skip this cell if you don't care about repeat-run speed.**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print("HF_HOME =", os.environ["HF_HOME"])

## 4. Launch the server in the background

Starts `python -m miniengine` as a subprocess, redirecting all logs to `server.log`. Then polls `/health` until the server is responsive. **The model will download on first run (~8 GB), so the first wait can take 3–5 minutes.**

Change `MODEL` to use a different one. Suggestions:
- `Qwen/Qwen3-0.6B-Instruct-2507` — fastest, for smoke tests
- `Qwen/Qwen3-4B-Instruct-2507` — README default, good for T4
- `Qwen/Qwen3-8B` — milestone-2 target, needs L4 or better

In [ ]:
import signal
import subprocess
import time

import requests

MODEL   = "Qwen/Qwen3-4B-Instruct-2507"  # @param {type:"string"}
DTYPE   = "bfloat16"                      # @param ["bfloat16", "float16", "float32"]
MODE    = "batched"                       # @param ["baseline", "batched"]
PORT    = 8000
MAX_RUN = 16

log_path = os.path.join(WORKDIR, "server.log")
log_f = open(log_path, "w")

server_proc = subprocess.Popen(
    [
        "python", "-u", "-m", "miniengine",
        "--model", MODEL,
        "--dtype", DTYPE,
        "--mode", MODE,
        "--host", "127.0.0.1",
        "--port", str(PORT),
        "--max-running", str(MAX_RUN),
    ],
    stdout=log_f,
    stderr=subprocess.STDOUT,
    start_new_session=True,  # so we can kill the whole group
    cwd=WORKDIR,
)
print(f"Server PID = {server_proc.pid}.  Logs \u2192 {log_path}")

BASE_URL = f"http://127.0.0.1:{PORT}"
DEADLINE = time.time() + 600  # 10-minute timeout for first-run download

while True:
    if server_proc.poll() is not None:
        print("\nServer crashed. Last 60 log lines:")
        print(subprocess.check_output(["tail", "-n", "60", log_path]).decode())
        raise RuntimeError("miniengine exited before becoming healthy")
    try:
        r = requests.get(f"{BASE_URL}/health", timeout=2)
        if r.status_code == 200:
            print("\nServer is healthy.")
            break
    except requests.exceptions.RequestException:
        pass
    if time.time() > DEADLINE:
        raise TimeoutError("Server did not become healthy within 10 minutes")
    print(".", end="", flush=True)
    time.sleep(5)

### Peek at the server log (handy while waiting / after a crash)

In [ ]:
!tail -n 40 server.log

## 5. Send a sample chat request

In [ ]:
import json

payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "In one sentence, explain why batched decoding speeds up an LLM serving engine."}
    ],
    "max_tokens": 128,
    "temperature": 0.0,
    "stream": True,
}

print("\nStreaming response:\n")
with requests.post(f"{BASE_URL}/v1/chat/completions", json=payload, stream=True, timeout=120) as resp:
    resp.raise_for_status()
    for raw in resp.iter_lines():
        if not raw:
            continue
        line = raw.decode("utf-8")
        if not line.startswith("data: "):
            continue
        data = line[len("data: "):]
        if data == "[DONE]":
            print("\n\n[stream finished]")
            break
        chunk = json.loads(data)
        delta = chunk["choices"][0].get("delta", {})
        piece = delta.get("content", "")
        print(piece, end="", flush=True)

## 6. (Optional) Quick serving benchmark

Tiny configuration so it finishes in a few minutes. For a real measurement, increase `--num-requests` and add more concurrency levels.

In [ ]:
!python -m benchmark.bench_serving \
    --input-len 256 --output-len 128 \
    --concurrencies 1,4,8 \
    --num-requests 8 \
    --randomness 0.5

## 7. (Optional) Quick accuracy benchmark

In [ ]:
!python -m benchmark.bench_accuracy --dataset mmlu --num-samples 50 --concurrency 4

## 8. Stop the server

Always run this when you're done so the GPU memory is released and the runtime is reusable.

In [ ]:
if server_proc.poll() is None:
    try:
        os.killpg(os.getpgid(server_proc.pid), signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        server_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        os.killpg(os.getpgid(server_proc.pid), signal.SIGKILL)
        server_proc.wait()
    print("Server stopped.")
else:
    print("Server already exited with code", server_proc.returncode)

log_f.close()